In [ ]:
# apktool.bat d D:\com.jvstudios.gpstracker-254.apk

Data sorting:
First, unzip the downloaded files. Then read each file and check whether it contains a file with the same APK name. There should be exactly two matching APK files. If there is only one, search in other directories for the second APK with the same name and move both together. If there are none, search in other directories for two matching APK files and move them together.

Data cleaning:
Traverse each folder and inspect the files inside to find APK files with the same name.

In [1]:
import os
import re
import pandas as pd
import subprocess
import time
from pathlib import Path

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
def extract_apk_info(root_dir):
    count = 0
    all_file = set()
    rs = set()
    non_rs = set()
    # Traverse the root directory and all its subdirectories.
    for root, dirs, files in os.walk(root_dir):
        # Get the current folder name as apk_name. (for example, ai.wizely.android)
        apk_name = os.path.basename(root)
        all_file.add(apk_name)
        # print(apk_name)
        
        # Ignore the root directory itself, and only process subdirectories with package names.
        if not apk_name or apk_name == os.path.basename(root_dir):
            continue
            
        for file in files:
            # Check if the file is an APK file and its name contains the apk_name (highly overlapping)
            if file.endswith('.apk') and apk_name in file:
                # print(file)
                rs.add(apk_name)
                count = count +1
    non_rs = all_file - rs 

    print(count)
    # print(non_rs)
    return non_rs

In [ ]:
SRC_DIR = Path("1122apk/raw/1071_apk_unzip") #Path("two_examples/raw") #Path("1122apk/raw/drive-download-20260305T173717Z-1-001")  # Folder containing the original APKs
non_rs = extract_apk_info(SRC_DIR)

2142


In [23]:
len(non_rs), non_rs

(1, {'1071_apk_unzip_1'})

The subdirectory name is the APK name. Each subdirectory should contain two APK files with the same name. If it does not, record that APK name.

In [ ]:
import os

def extract_apk_info(root_dir):
    invalid_apk_names = []
    valid_apk_names = []
    total_apk_dirs = 0

    # Only traverse the first-level subdirectories of root_dir
    for apk_name in os.listdir(root_dir):
        apk_dir = os.path.join(root_dir, apk_name)
        # print(apk_dir)
        # Only process subdirectories
        if not os.path.isdir(apk_dir):
            continue

        total_apk_dirs += 1

        # All .apk files in the current subdirectory with filenames starting with "apk_name"
        matched_apks = [
            f for f in os.listdir(apk_dir)
            if os.path.isfile(os.path.join(apk_dir, f))
            and f.endswith(".apk")
            and f.startswith(apk_name)
        ]

        if len(matched_apks) == 2:
            valid_apk_names.append(apk_name)
        else:
            invalid_apk_names.append(apk_name)
            print(f"[INVALID] {apk_name}: found {len(matched_apks)} apk(s) -> {matched_apks}")

    print(f"Total number of subdirectories: {total_apk_dirs}")
    print(f"Number of directories meeting the condition (exactly two APKs with the same name): {len(valid_apk_names)}")
    print(f"Number of directories not meeting the condition: {len(invalid_apk_names)}")

    return invalid_apk_names

In [ ]:
root_dir = r"1122apk/raw/1071_apk_unzip"
invalid_apk_names = extract_apk_info(root_dir)

In [28]:
invalid_apk_names

['com.iz.games.preschool.game.kindergarten.baby.children.kids.learning.educational.learn.toddler.puzzles.stories.coloring',
 'com.iz.unicorn.coloring.book.games.glitter.free.colouring.pages.pony.princess.animal.drawing.girls.color',
 'com.lonelycatgames.Xplore',
 'com.sriandroid.justkannada',
 'com.voicesms.writesmsbyvoice.theburraq',
 'com.whindipanchangcalendar.iwebnapp',
 'document.scannerapp.docscannerapp.android.imagescanner.free.camscanner.documentscanner.pdfscanner.textscanner.ocr',
 'face.makeup.beauty.photoeditor',
 'hairstyles.hairstylesstepbystep.schoolhairstyles.hairstyle2.hairstylesgirls.hairstylestepbystep.hairstyle']

Use apktool to decompress the APKs in the specified directory into the APK directory.

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Does not rely on the process exit signal.

As long as the output directory becomes stable, continue to the next task — if the file count or total size stops changing for N seconds, it is considered complete.

Large APKs will automatically wait longer.

After timeout, the process tree will be killed to avoid leaving any java.exe processes behind.

In [6]:
import os
import re
import pandas as pd
import subprocess
import time
from pathlib import Path

In [ ]:
def dir_snapshot(out_dir: Path):
    """Return (number of files, total bytes) as a progress snapshot."""
    n = 0
    total = 0
    if not out_dir.exists():
        return 0, 0
    for p in out_dir.rglob("*"):
        if p.is_file():
            n += 1
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return n, total

def looks_done(out_dir: Path):
    """Quick Sentry: Matches found in manifest or apktool.yml."""
    return (out_dir / "AndroidManifest.xml").exists() and (out_dir / "apktool.yml").exists()

def run_apktool_with_completion_guard(APKTOOL, apk_path: Path, out_dir: Path,
                                      short_guard_s=30, long_guard_s=180,
                                      stable_window_s=8, poll_s=2):
    """
    - First, allow `short_guard_s` time: if the directory is stable/complete but the process has not exited, issue a `kill` command to let it proceed.
    - If not complete, allow `long_guard_s` time: apply the same logic.
    - Returns: status string; `cmd = [APKTOOL, "d", str(apk_path), "-f", "-o", str(output_path)]`
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = ["cmd.exe", "/c", APKTOOL, "d", str(apk_path), "-f", "-o", str(out_dir)]
    p = subprocess.Popen(
        cmd,
        stdout=None,
        stderr=None,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP
    )

    def wait_until(limit_s):
        start = time.time()
        last_change = time.time()
        last_snap = dir_snapshot(out_dir)

        while True:
            time.sleep(poll_s)

            # If the process exits on its own, it terminates immediately.
            rc = p.poll()
            if rc is not None:
                return "exited", rc

            # If the output directory is still changing, update the last change time.
            snap = dir_snapshot(out_dir)
            if snap != last_snap:
                last_snap = snap
                last_change = time.time()

            # Completion is considered achieved if the "completion sentinel" condition is met and the directory has remained stable for a period of time.
            if looks_done(out_dir) and (time.time() - last_change) >= stable_window_s:
                return "done_but_hanging", None

            # time out
            if (time.time() - start) >= limit_s:
                return "timeout", None

    # Section 1: Short-term Insurance
    reason, rc = wait_until(short_guard_s)
    if reason == "exited" and rc == 0:
        return "ok_fast_exit"
    if reason == "done_but_hanging":
        subprocess.run(["taskkill", "/PID", str(p.pid), "/T", "/F"], capture_output=True, text=True)
        return "ok_fast_hang_killed"
    if reason == "timeout":
        # Not yet complete: Entering a long wait (do not kill it immediately).
        pass
    elif reason == "exited":
        # rc != 0 In the event of failure, it also enters a long wait before retrying once (depending on your needs).
        pass

    # Second Stage: Long Wait
    reason2, rc2 = wait_until(long_guard_s)
    if reason2 == "exited" and rc2 == 0:
        return "ok_long_exit"
    if reason2 == "done_but_hanging":
        subprocess.run(["taskkill", "/PID", str(p.pid), "/T", "/F"], capture_output=True, text=True)
        return "ok_long_hang_killed"

    # Still incomplete or failed: Terminate and return failure.
    subprocess.run(["taskkill", "/PID", str(p.pid), "/T", "/F"], capture_output=True, text=True)
    return f"{apk_path}_failed_{reason2}"

In [ ]:
def parse_file_info(path):
    # 1. Get file name: com.mhbl.sastasundar-176.ap
    filename = os.path.basename(path)
    
    # 2. Remove the file extension (regardless of whether it is .ap, .txt, or .apk)
    # splitext only strips off the last dot and everything following it 
    name_without_ext = os.path.splitext(filename)[0]
    
    # 3. Extract apk_name and version.
    if '-' in name_without_ext:
        # `rsplit('-', 1)` ensures the split occurs only once, at the rightmost hyphen.
        parts = name_without_ext.rsplit('-', 1)
        apk_name = parts[0]   # com.mhbl.sastasundar
        version = parts[1]    # 176
    else:
        apk_name = name_without_ext
        version = "Unknown"
        
    return apk_name, version
# apk, ver = parse_file_info("com.mhbl.sastasundar-176.apk")
# apk, ver

In [9]:
# [1122apk/raw/1071_apk_unzip/AA2_second_time/AA_first_100_batch
# 1122apk/raw/1071_apk_unzip/AA2_second_time/AA_second_100_batch
# 1122apk/raw/1071_apk_unzip/AA2_second_time/AA_third_100_batch
# 1122apk/raw/1071_apk_unzip/AA2_second_time/AA_fourth_50_batch
# ]

# [
#     1122apk/dicompile1122apk/AA2_second_batch/AA1_first_100_batch
#     1122apk/dicompile1122apk/AA2_second_batch/AA3_third_100_batch
#     1122apk/dicompile1122apk/AA2_second_batch/AA5_fifth_100_batch
#     1122apk/dicompile1122apk/AA2_second_batch/AA7_seventh_100_batch
# ]

### AA3_third_time
# [1122apk/raw/1071_apk_unzip/AA3_third_time/AA_first_100_batch
# 1122apk/raw/1071_apk_unzip/AA3_third_time/AA_second_100_batch
# 1122apk/raw/1071_apk_unzip/AA3_third_time/AA_third_100_batch
# 1122apk/raw/1071_apk_unzip/AA3_third_time/AA_fourth_50_batch
# ]

# [
#     1122apk/dicompile1122apk/AA3_third_batch/AA1_first_100_batch
#     1122apk/dicompile1122apk/AA3_third_batch/AA3_third_100_batch
#     1122apk/dicompile1122apk/AA3_third_batch/AA5_fifth_100_batch
#     1122apk/dicompile1122apk/AA3_third_batch/AA7_seventh_100_batch
# ]

### AA4_forth_time
# 1122apk/raw/1071_apk_unzip/AA4_fourth_time

# 1122apk/dicompile1122apk/AA4_fourth_batch

In [ ]:
# 1. Define the tool path and directories
# Recommended style
APKTOOL = r"D:\softwall_install\apktool\apktool.bat"
SRC_DIR = Path("1122apk\\raw\\1071_apk_unzip\\AA4_fourth_time")#Path("1122apk/raw/1071_apk_unzip") #Path("1122apk/raw/50apk/4_leftover_app") #Path("1122apk/raw/drive-download-20260305T173717Z-1-001")  # Directory storing the original APKs Path("1122apk/raw/50apk/4_leftover_app") Path("two_examples/raw")
DEST_DIR = Path("1122apk/1122apk_privacy_policy_url/AA4_fourth_batch") #Path("1122apk/dicompile1122apk")#Path("1122apk/dicompile1122apk") #Path("1122apk/dicompile1122apk") # Path("1122apk/test/")      # Output directory for decompiled APKs Path("two_examples/dicompile")

# Create the output directory if it does not already exist
DEST_DIR.mkdir(parents=True, exist_ok=True)
count = 1
results_container = []

for root, dirs, files in os.walk(SRC_DIR):
    root_path = Path(root)
    # print("root_path", root_path)
    # Get the current folder name as apk_name (for example, ai.wizely.android)
    apk_name = os.path.basename(root)
    # print(apk_name)

    
    # Ignore the root directory itself and only process subdirectories with package names
    if not apk_name or apk_name == os.path.basename(SRC_DIR):
        continue
        
    # if apk_name == "amazon.shop.barcode.scanner":
    for file in files:
        # Check whether the file is an APK and whether the filename matches apk_name closely
        if file.endswith('.apk') and apk_name in file:
            # if count == 2:
                # break
            print(f"Found {file}, preparing to start...")
            # # 3. Loop through apktool to decompile
            apk_path = root_path / file
            # print('apk_path', apk_path)

            
            # Set the output directory name for the decompiled result
            output_path = DEST_DIR / Path(file).stem
            # Create the output directory if it does not exist
            output_path.mkdir(parents=True, exist_ok=True)

            print(f"\n==> Starting: {file}")
            print(f"    APK : {apk_path}")
            print(f"    OUT : {output_path}")

            status = run_apktool_with_completion_guard(APKTOOL, apk_path, output_path,
                                        short_guard_s=30,
                                        long_guard_s=200,
                                        stable_window_s=8)
            print(apk_path.name, status)

            apk, version = parse_file_info(file)

            row = {
                "apk_name": apk_name,
                "version": version,
                "status": status # If missing, this will be None (JSON null)
            }
            results_container.append(row)

            count = count + 1

# 4. Convert to a DataFrame in one step
df = pd.DataFrame(results_container)

# 5. Save the results
df.to_csv(f"{DEST_DIR}/dicompile_apks_batch_2", index=False, encoding='utf-8-sig')

print("\n--- All tasks completed ---")

apktool has actually already extracted all the files, but the process still does not exit.